# 03 • Métriques, calibration et seuils

`[MÉTA | Formation 4-024 | Niveau Application | TP 03 | Mode CPU local]`

**Objectif :** Choisir une mesure utile et un seuil sur validation seulement.

**Temps indicatif :** 30 min et réutilisation J3. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** TP 02.

**Preuves de réussite :** Matrice interprétée, AP distinguée de PR-AUC, capacité chiffrée.

**Sources :** R04.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [1]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


Moteur disponible : 2.10.0+cpu | Données : /mnt/data/deep_learning_4_024/03_Travaux_pratiques/donnees


## 1. Scores de référence
On reprend un modèle linéaire rapide sur les mêmes données. Les principes d’évaluation ne dépendent pas du choix neuronal.

In [2]:
d=split_tabular();m=LogisticRegression(max_iter=300).fit(*d['train'])
X,y=d['validation'];score=m.predict_proba(X)[:,1]
from sklearn.metrics import ConfusionMatrixDisplay,precision_recall_curve,roc_curve,auc
print(pd.DataFrame([binary_metrics(y,score,t) for t in [.2,.5,.8]]).round(3))
ConfusionMatrixDisplay.from_predictions(y,score>=.5,display_labels=['non','oui'])
plt.title('Validation : seuil 0,5');plt.tight_layout();plt.savefig(RESULTS/'03_confusion.png',dpi=150)

   seuil  exactitude  precision  ...  precision_moyenne_AP  brier  alertes
0    0.2       0.925      0.614  ...                 0.732  0.048       44
1    0.5       0.939      0.857  ...                 0.732  0.048       21
2    0.8       0.931      1.000  ...                 0.732  0.048       12

[3 rows x 9 columns]


## 2. Courbe précision-rappel et AP
L’average precision (AP, précision moyenne) n’est pas strictement l’aire trapézoïdale sous la courbe précision-rappel. Afficher les deux noms exactement pour éviter une confusion dans la restitution. La ROC n’est pas supprimée ; la courbe PR répond à une autre question utile aux classes rares.

In [3]:
p,r,thresholds=precision_recall_curve(y,score)
ap=average_precision_score(y,score);pr_auc=auc(r,p)
fig,ax=plt.subplots(figsize=(7,4));ax.plot(r,p);ax.set(xlabel='Rappel',ylabel='Précision',title=f'Validation : AP={ap:.3f} ; aire trapézoïdale={pr_auc:.3f}')
fig.tight_layout();fig.savefig(RESULTS/'03_precision_rappel.png',dpi=150)
print({'AP':ap,'PR_AUC_trapezoidale':pr_auc,'prevalence':float(y.mean())})

{'AP': 0.7321897838927701, 'PR_AUC_trapezoidale': 0.7301983124701333, 'prevalence': 0.10277777777777777}


## 3. Choisir un seuil opérationnel
Contrainte fictive : au plus 50 alertes sur cette validation. Chercher parmi les seuils des scores disponibles celui maximisant le rappel dans cette capacité. En cas d’égalité, privilégier le seuil le plus élevé. Ce choix ne garantit pas 50 alertes dans un futur lot de taille ou de distribution différente.

In [4]:
candidats=sorted(set(score),reverse=True)+[1.000001]
rows=[binary_metrics(y,score,t) for t in candidats]
admissibles=[row for row in rows if row['alertes']<=50]
choix=max(admissibles,key=lambda row:(row['rappel'],row['seuil']))
print('Seuil choisi sur validation :',choix)
assert choix['alertes']<=50
print('Classement top-50 :',topk_metrics(y,score,50))

Seuil choisi sur validation : {'seuil': 0.16389430914287184, 'exactitude': 0.9222222222222223, 'precision': 0.5957446808510638, 'rappel': 0.7567567567567568, 'f1': 0.6666666666666666, 'roc_auc': 0.8671240900343068, 'precision_moyenne_AP': 0.7321897838927701, 'brier': 0.04780716916831159, 'alertes': 47}
Classement top-50 : {'k': 50, 'cas_pertinents': 28, 'precision_a_k': 0.56, 'rappel_a_k': 0.7567567567567568}


## 4. Calibration
Comparer fréquence observée et score moyen par intervalle. Des intervalles peu peuplés sont instables. Le graphique décrit ce modèle et ce jeu, sans garantir des probabilités fiables en production.

In [5]:
from sklearn.calibration import calibration_curve
observes,predits=calibration_curve(y,score,n_bins=6,strategy='quantile')
fig,ax=plt.subplots(figsize=(6,4));ax.plot(predits,observes,marker='o',label='Modèle');ax.plot([0,1],[0,1],linestyle=':',label='Référence idéale')
ax.set(xlabel='Score moyen',ylabel='Fréquence positive',title='Fiabilité sur validation');ax.legend();fig.tight_layout();fig.savefig(RESULTS/'03_calibration.png',dpi=150)
save_result('03_metr iques'.replace(' ',''),{'AP':ap,'PR_AUC_trapezoidale':pr_auc,'seuil_validation':choix,'test_ouvert':False})

PosixPath('/mnt/data/deep_learning_4_024/03_Travaux_pratiques/resultats/03_metriques.json')

## 5. Questions à restituer
Pourquoi l’accuracy peut-elle être trompeuse ? Qu’est-ce qu’une alerte utile ? Que change le seuil, et que ne change-t-il pas ? Pourquoi la calibration et le classement ne sont-ils pas synonymes ? **Extension :** ajouter un coût fictif explicite aux faux positifs et faux négatifs, puis comparer au critère de capacité.